In [32]:
from extractor import extract_file, render_and_save
from gpt_api_wrapper import BatchConversation
from pathlib import Path
import datetime

In [33]:
prompts_extracted = extract_file("/mnt/c/Users/sanps/Desktop/Projects/eai/eai/eai/eai_starter_kit/llm_prompts/behavior_transition_modeling_prompts.json")

In [34]:
first = prompts_extracted[0]
print(first)

ExtractionResult(identifier='assembling_gift_baskets_0_Beechwood_0_int_0_2021-10-26_12-46-37', prompt_type='behavior_transition_modeling', dynamic_content={'problem_file': '(define (problem assembling_gift_baskets)\n    (:domain igibson)\n    (:objects agent_n_01_1 - agent basket_n_01_1 basket_n_01_2 - basket_n_01 bow_n_08_2 - bow_n_08 candle_n_01_1 - candle_n_01 cheese_n_01_1 - cheese_n_01 cookie_n_01_1 - cookie_n_01 floor_n_01_1 - floor_n_01 table_n_02_1 table_n_02_2 - table_n_02)\n    (:init (onfloor basket_n_01_1 floor_n_01_1) (onfloor basket_n_01_2 floor_n_01_1) (ontop bow_n_08_2 table_n_02_2) (ontop candle_n_01_1 table_n_02_1) (ontop cheese_n_01_1 table_n_02_2) (ontop cookie_n_01_1 table_n_02_1) (same_obj basket_n_01_1 basket_n_01_1) (same_obj basket_n_01_2 basket_n_01_2) (same_obj bow_n_08_2 bow_n_08_2) (same_obj candle_n_01_1 candle_n_01_1) (same_obj cheese_n_01_1 cheese_n_01_1) (same_obj cookie_n_01_1 cookie_n_01_1) (same_obj floor_n_01_1 floor_n_01_1) (same_obj table_n_02_1 t

In [35]:
first_dict = first.as_dict()
first_dict

{'identifier': 'assembling_gift_baskets_0_Beechwood_0_int_0_2021-10-26_12-46-37',
 'prompt_type': 'behavior_transition_modeling',
 'dynamic_content': {'problem_file': '(define (problem assembling_gift_baskets)\n    (:domain igibson)\n    (:objects agent_n_01_1 - agent basket_n_01_1 basket_n_01_2 - basket_n_01 bow_n_08_2 - bow_n_08 candle_n_01_1 - candle_n_01 cheese_n_01_1 - cheese_n_01 cookie_n_01_1 - cookie_n_01 floor_n_01_1 - floor_n_01 table_n_02_1 table_n_02_2 - table_n_02)\n    (:init (onfloor basket_n_01_1 floor_n_01_1) (onfloor basket_n_01_2 floor_n_01_1) (ontop bow_n_08_2 table_n_02_2) (ontop candle_n_01_1 table_n_02_1) (ontop cheese_n_01_1 table_n_02_2) (ontop cookie_n_01_1 table_n_02_1) (same_obj basket_n_01_1 basket_n_01_1) (same_obj basket_n_01_2 basket_n_01_2) (same_obj bow_n_08_2 bow_n_08_2) (same_obj candle_n_01_1 candle_n_01_1) (same_obj cheese_n_01_1 cheese_n_01_1) (same_obj cookie_n_01_1 cookie_n_01_1) (same_obj floor_n_01_1 floor_n_01_1) (same_obj table_n_02_1 table_

In [36]:
def get_action_name(action_str: str) -> str:
    line1 = action_str.split("\n")[0].strip()
    action_name = line1[len("(:action "):-1]
    return action_name
print(get_action_name(first_dict['dynamic_content']['actions'][1]))

open


In [37]:
prompt_dicts = [p.as_dict() for p in prompts_extracted]

In [38]:
# add key 'action_names' to prompt_dicts

action_set = set()

for entry in prompt_dicts:
    action_names = []
    for action_str in entry['dynamic_content']['actions']:
        action_name = get_action_name(action_str)
        action_names.append(action_name)
        action_set.add(action_name)
    entry['dynamic_content']['action_names'] = action_names
print(f"Total unique action names: {len(action_set)}")
    

Total unique action names: 28


In [39]:
# load data/gold_action.json

with open("/mnt/c/Users/sanps/Desktop/Projects/eai/eai/eai/scripts/experiments/b_tm/ref_answer/data/gold_action.json", "r") as f:
    import json
    gold_data = json.load(f)
print(len(gold_data))

30


In [40]:
# check set diff
gold_action_set = set([key for key in gold_data.keys()])
diff1 = action_set - gold_action_set
diff2 = gold_action_set - action_set
print(f"Actions in prompts but not in gold data: {diff1}")
print(f"Actions in gold data but not in prompts: {diff2}")

Actions in prompts but not in gold data: {'slice_carvingknife'}
Actions in gold data but not in prompts: {'slice-carvingknife', 'place_ontop', 'freeze'}


In [41]:
def get_matching_gold_entry(action_name: str):
    if action_name in gold_data:
        return gold_data[action_name]
    else:
        # replace underscores with hyphens and check again
        alt_name = action_name.replace("_", "-")
        if alt_name in gold_data:
            return gold_data[alt_name]
    return None


In [42]:
# print fields of a random action from gold

random_action = list(action_set)[0]
matching_entry = get_matching_gold_entry(random_action)
print(f"Matching entry for action '{random_action}': {matching_entry}")
print(f"Total entries in gold data: {len(gold_data)}")
print(f"Total unique action names extracted from prompts: {len(action_set)}")

Matching entry for action 'slice_carvingknife': {'action_name': 'slice-carvingknife', 'action_parameters': '(?obj - object ?knife - carving_knife_n_01 ?board - countertop_n_01 ?agent - agent)', 'action_preconditions': '(and (in_reach_of_agent ?obj) (holding ?knife) (ontop ?obj ?board) (not (sliced ?obj)))', 'action_effects': '(sliced ?obj)'}
Total entries in gold data: 30
Total unique action names extracted from prompts: 28


In [43]:
# data context: Matching entry for action 'grasp': {'action_name': 'grasp', 'action_parameters': '(?obj - object ?agent - agent)', 'action_preconditions': '(and (not (holding ?obj)) (not (handsfull ?agent)) (in_reach_of_agent ?obj) (not (exists (?obj2 - object) (and (inside ?obj ?obj2) (not (open ?obj2))))))', 'action_effects': '(and (holding ?obj) (handsfull ?agent) (forall (?other_obj - object) (and (not (inside ?obj ?other_obj)) (not (ontop ?obj ?other_obj)) (not (under ?obj ?other_obj)) (not (under ?other_obj ?obj)) (not (nextto ?obj ?other_obj)) (not (nextto ?other_obj ?obj)) (not (onfloor ?obj ?other_obj)))))'}
# required format example:             "output": "(:action navigate_to\n:parameters (?objto - object ?agent - agent)\n:precondition (and (not (in_reach_of_agent ?objto)))\n:effect (and (in_reach_of_agent ?objto) (forall (?obj - object) (when (and (in_reach_of_agent ?obj) (not (same_obj ?obj ?objto))) (not (in_reach_of_agent ?obj)))))\n)\n(:action open\n:parameters (?obj - object ?agent - agent)\n:precondition (and (in_reach_of_agent ?obj) (not (open ?obj)))\n:effect (and (open ?obj))\n)\n(:action place_inside\n:parameters (?obj_in_hand - object ?obj - object ?agent - agent)\n:precondition (and (holding ?obj_in_hand) (in_reach_of_agent ?obj))\n:effect (and (inside ?obj_in_hand ?obj) (not (holding ?obj_in_hand)))\n)\n(:action grasp\n:parameters (?obj - object ?agent - agent)\n:precondition (and (in_reach_of_agent ?obj) (not (holding ?obj)))\n:effect (and (holding ?obj))\n)"


format_template = """(:action {action_name}
:parameters {action_parameters}
:precondition {action_preconditions}
:effect {action_effects}
)
"""

# add 'formatted_output' field to prompt_dicts
for entry in prompt_dicts:
    formatted_outputs = []
    for action_name in entry['dynamic_content']['action_names']:
        matching_entry = get_matching_gold_entry(action_name)
        if matching_entry is not None:
            formatted_output = format_template.format(
                action_name=action_name,
                action_parameters=matching_entry['action_parameters'],
                action_preconditions=matching_entry['action_preconditions'],
                action_effects=matching_entry['action_effects']
            )
            formatted_outputs.append(formatted_output)
    entry['formatted_output'] = "\n".join(formatted_outputs)
    
# print formatted output of first prompt
print(prompt_dicts[0]['formatted_output'])


(:action navigate_to
:parameters (?objto - object ?agent - agent)
:precondition (not (in_reach_of_agent ?objto))
:effect (and (in_reach_of_agent ?objto) (forall (?objfrom - object) (when (and (in_reach_of_agent ?objfrom) (not (same_obj ?objfrom ?objto))) (not (in_reach_of_agent ?objfrom)))))
)

(:action open
:parameters (?obj - object ?agent - agent)
:precondition (and (in_reach_of_agent ?obj) (not (open ?obj)) (not (handsfull ?agent)))
:effect (open ?obj)
)

(:action place_inside
:parameters (?obj_in_hand - object ?obj - object ?agent - agent)
:precondition (and (holding ?obj_in_hand) (in_reach_of_agent ?obj) (open ?obj))
:effect (and (inside ?obj_in_hand ?obj) (not (holding ?obj_in_hand)) (not (handsfull ?agent)))
)

(:action grasp
:parameters (?obj - object ?agent - agent)
:precondition (and (not (holding ?obj)) (not (handsfull ?agent)) (in_reach_of_agent ?obj) (not (exists (?obj2 - object) (and (inside ?obj ?obj2) (not (open ?obj2))))))
:effect (and (holding ?obj) (handsfull ?agent

In [44]:
org_prompt_path = Path("/mnt/c/Users/sanps/Desktop/Projects/eai/eai/eai/eai_starter_kit/llm_prompts/behavior_transition_modeling_prompts.json")

# load
with open(org_prompt_path, "r") as f:
    import json
    org_prompts = json.load(f)
first_org_prompt = org_prompts[0]['llm_prompt']
print(first_org_prompt)


The following is predicates defined in this domain file. Pay attention to the types for each predicate.
(define (domain igibson)

    (:requirements :strips :adl :typing :negative-preconditions)

    (:types 
        vacuum_n_04 facsimile_n_02 dishtowel_n_01 apparel_n_01 seat_n_03 bottle_n_01 mouse_n_04 window_n_01 scanner_n_02 
        sauce_n_01 spoon_n_01 date_n_08 egg_n_02 cabinet_n_01 yogurt_n_01 parsley_n_02 notebook_n_01 dryer_n_01 saucepan_n_01 
        soap_n_01 package_n_02 headset_n_01 fish_n_02 vehicle_n_01 chestnut_n_03 grape_n_01 wrapping_n_01 makeup_n_01 mug_n_04 
        pasta_n_02 beef_n_02 scrub_brush_n_01 cracker_n_01 flour_n_01 sunglass_n_01 cookie_n_01 bed_n_01 lamp_n_02 food_n_02 
        painting_n_01 carving_knife_n_01 pop_n_02 tea_bag_n_01 sheet_n_03 tomato_n_01 agent_n_01 hat_n_01 dish_n_01 cheese_n_01 
        perfume_n_02 toilet_n_02 broccoli_n_02 book_n_02 towel_n_01 table_n_02 pencil_n_01 rag_n_01 peach_n_03 water_n_06 cup_n_01 
        radish_n_01 marker

In [45]:
import random
assert len(org_prompts) == len(prompt_dicts)

In [46]:
# check if identifiers match

for i in range(len(org_prompts)):
    assert org_prompts[i]['identifier'] == prompt_dicts[i]['identifier'], f"Mismatch at index {i}: {org_prompts[i]['identifier']} != {prompt_dicts[i]['identifier']}"
print("All identifiers match.")

All identifiers match.


In [47]:
def get_answer_dp(dict_entry):
    return {
        "identifier": dict_entry['identifier'],
        "prompt": org_prompts[0]['llm_prompt'],
        "llm_output": {
            "output": dict_entry['formatted_output']
        }
    }
submission_data = [get_answer_dp(entry) for entry in prompt_dicts]
print(submission_data[0])

{'identifier': 'assembling_gift_baskets_0_Beechwood_0_int_0_2021-10-26_12-46-37', 'prompt': '\nThe following is predicates defined in this domain file. Pay attention to the types for each predicate.\n(define (domain igibson)\n\n    (:requirements :strips :adl :typing :negative-preconditions)\n\n    (:types \n        vacuum_n_04 facsimile_n_02 dishtowel_n_01 apparel_n_01 seat_n_03 bottle_n_01 mouse_n_04 window_n_01 scanner_n_02 \n        sauce_n_01 spoon_n_01 date_n_08 egg_n_02 cabinet_n_01 yogurt_n_01 parsley_n_02 notebook_n_01 dryer_n_01 saucepan_n_01 \n        soap_n_01 package_n_02 headset_n_01 fish_n_02 vehicle_n_01 chestnut_n_03 grape_n_01 wrapping_n_01 makeup_n_01 mug_n_04 \n        pasta_n_02 beef_n_02 scrub_brush_n_01 cracker_n_01 flour_n_01 sunglass_n_01 cookie_n_01 bed_n_01 lamp_n_02 food_n_02 \n        painting_n_01 carving_knife_n_01 pop_n_02 tea_bag_n_01 sheet_n_03 tomato_n_01 agent_n_01 hat_n_01 dish_n_01 cheese_n_01 \n        perfume_n_02 toilet_n_02 broccoli_n_02 book_n

In [48]:
len(submission_data[0]['prompt'] + submission_data[0]['llm_output']['output'])

12429

In [49]:
# add field str of llm_output 
for entry in submission_data:
    entry['llm_output_str'] = str(entry['llm_output'])

In [50]:
# save in ../data/b_tm.jsonl
import json
import jsonlines
with jsonlines.open("../data/b_tm.jsonl", mode='w') as writer:
    for entry in submission_data:
        writer.write(entry)
        